In [1]:
import pandas as pd
import requests
import numpy as np
from lightweight_charts import Chart
from stock_indicators import indicators, Quote
from datetime import datetime, timedelta
import asyncio
import nest_asyncio

nest_asyncio.apply()

In [2]:
# Add required imports
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
import yfinance as yf
df = yf.download('SPXL', start='2010-01-01', multi_level_index=False)
df.reset_index(inplace=True)
df.to_csv('SPXL.csv', index=False)
df = pd.read_csv('SPXL.csv')
rawdf = df.copy()
df['Date'] = pd.to_datetime(df['Date'])
df.head()

C:\Users\jwang\AppData\Local\Temp\ipykernel_14952\2805507071.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download('SPXL', start='2010-01-01', multi_level_index=False)
[*********************100%***********************]  1 of 1 completed


,Date,Close,High,Low,Open,Volume
0,2010-01-04,4.081054,4.087708,3.983463,3.984202,28238400
1,2010-01-05,4.119499,4.121717,4.029302,4.076618,33206400
2,2010-01-06,4.130589,4.157204,4.098059,4.107670,44194800
3,2010-01-07,4.179383,4.192691,4.069225,4.109148,43773600
4,2010-01-08,4.223743,4.228919,4.119499,4.143157,39685200


In [4]:
quotes = [
    Quote(d, o, h, l, c, v)
    for d, o, h, l, c, v in zip(
        df['Date'],
        df['Open'],
        df['High'],
        df['Low'],
        df['Close'],
        df['Volume']
    )
]


In [5]:
# ATR Trailing Stop
df['atr_stop'] = [r.atr_stop for r in indicators.get_atr_stop(quotes)]
df['atr_stop'] = df['atr_stop'].astype(float) 
df.to_json()
df['bullishATRStop'] = 0.0
df['bullishATRStop'] = np.where(df['Close'] > df['atr_stop'], 1.0, 0.0)
df['crossover_ATRStop'] = df['bullishATRStop'].diff()



In [6]:
df.tail()

,Date,Close,High,Low,Open,Volume,atr_stop,bullishATRStop,crossover_ATRStop
3974,2025-10-21,214.619995,216.369995,213.500000,214.880005,2023400,196.078320,1.0,0.0
3975,2025-10-22,211.300003,215.339996,206.970001,215.339996,3336000,196.078320,1.0,0.0
3976,2025-10-23,214.839996,215.929993,211.360001,211.559998,2436700,196.221948,1.0,0.0
3977,2025-10-24,220.000000,221.279999,218.610001,219.350006,2482600,201.348525,1.0,0.0
3978,2025-10-27,227.860001,228.179993,224.850006,225.460007,4089500,208.928121,1.0,0.0


In [7]:
# Convert supertrend to numeric for calculations
df['atr_stop'] = pd.to_numeric(df['atr_stop'], errors='coerce')

# Initialize columns for backtesting
df['Position'] = 0  # 1 for long, 0 for no position, -1 for short
df['Signal'] = df['crossover_ATRStop']  # Buy (1) or Sell (-1) signals
df['Entry_Price'] = 0.0
df['Exit_Price'] = 0.0
df['Trade_Return'] = 0.0
df['Cumulative_Return'] = 0.0

# Initialize variables for tracking trades
current_position = 0
entry_price = 0
initial_capital = 100000  # Starting with $100,000
capital = initial_capital
trade_history = []

# Perform backtesting
for i in range(1, len(df)):
    signal = df.iloc[i]['Signal']
    current_price = df.iloc[i]['Close']
    
    # Buy Signal
    if signal == 1 and current_position == 0:
        current_position = 1
        entry_price = current_price
        df.loc[df.index[i], 'Position'] = 1
        df.loc[df.index[i], 'Entry_Price'] = entry_price
        trade_history.append({
            'Date': df.iloc[i]['Date'],
            'Type': 'Buy',
            'Price': entry_price,
            'Capital': capital
        })
    
    # Sell Signal
    elif signal == -1 and current_position == 1:
        exit_price = current_price
        returns = (exit_price - entry_price) / entry_price
        capital *= (1 + returns)
        df.loc[df.index[i], 'Position'] = 0
        df.loc[df.index[i], 'Exit_Price'] = exit_price
        df.loc[df.index[i], 'Trade_Return'] = returns
        current_position = 0
        trade_history.append({
            'Date': df.iloc[i]['Date'],
            'Type': 'Sell',
            'Price': exit_price,
            'Capital': capital
        })

    # Update cumulative returns
    df.loc[df.index[i], 'Cumulative_Return'] = (capital / initial_capital - 1) * 100

# Convert trade history to DataFrame
trade_df = pd.DataFrame(trade_history)

# Calculate performance metrics
total_trades = len(trade_df) // 2  # Divide by 2 since each complete trade has a buy and sell
winning_trades = len(df[df['Trade_Return'] > 0])
losing_trades = len(df[df['Trade_Return'] < 0])
win_rate = winning_trades / total_trades if total_trades > 0 else 0
final_return = (capital / initial_capital - 1) * 100
max_drawdown = (df['Cumulative_Return'].cummax() - df['Cumulative_Return']).max()

In [8]:
# Plot equity curve
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                   vertical_spacing=0.03, subplot_titles=('Price with ATR Trailing Stop', 'Equity Curve'),
                   row_heights=[0.7, 0.3])

# Add candlestick
fig.add_trace(go.Candlestick(x=df['Date'],
                            open=df['Open'],
                            high=df['High'],
                            low=df['Low'],
                            close=df['Close'],
                            name='OHLC'),
              row=1, col=1)

# Add ATR Trailing Stop line
fig.add_trace(go.Scatter(x=df['Date'],
                        y=df['atr_stop'],
                        mode='lines',
                        name='ATR Trailing Stop',
                        line=dict(color='yellow')),
              row=1, col=1)

# Add buy signals
buy_signals = df[df['crossover_ATRStop'] == 1]
fig.add_trace(go.Scatter(x=buy_signals['Date'],
                        y=buy_signals['Low'],
                        mode='markers',
                        name='Buy Signal',
                        marker=dict(color='green', symbol='triangle-up', size=10)),
              row=1, col=1)

# Add sell signals
sell_signals = df[df['crossover_ATRStop'] == -1]
fig.add_trace(go.Scatter(x=sell_signals['Date'],
                        y=sell_signals['High'],
                        mode='markers',
                        name='Sell Signal',
                        marker=dict(color='red', symbol='triangle-down', size=10)),
              row=1, col=1)

# Add equity curve
fig.add_trace(go.Scatter(x=df['Date'],
                        y=df['Cumulative_Return'],
                        mode='lines',
                        name='Equity Curve',
                        line=dict(color='blue')),
              row=2, col=1)

# Update layout
fig.update_layout(
    title_text="ATR Trailing Stop Strategy Backtest Results",
    xaxis_rangeslider_visible=False,
    height=800
)

fig.show()

In [9]:
# Print performance metrics
print(f"Performance Metrics (2010-2025):")
print(f"--------------------------------")
print(f"Initial Capital: ${initial_capital:,.2f}")
print(f"Final Capital: ${capital:,.2f}")
print(f"Total Return: {final_return:.2f}%")
print(f"Total Trades: {total_trades}")
print(f"Winning Trades: {winning_trades}")
print(f"Losing Trades: {losing_trades}")
print(f"Win Rate: {win_rate:.2%}")
print(f"Maximum Drawdown: {max_drawdown:.2f}%")

Performance Metrics (2010-2025):
--------------------------------
Initial Capital: $100,000.00
Final Capital: $1,040,692.77
Total Return: 940.69%
Total Trades: 71
Winning Trades: 35
Losing Trades: 36
Win Rate: 49.30%
Maximum Drawdown: 230.64%


In [10]:
# Calculate Buy and Hold Returns
initial_price = df.iloc[0]['Close']
final_price = df.iloc[-1]['Close']
buy_hold_return = ((final_price - initial_price) / initial_price) * 100

print("\nBuy and Hold Strategy:")
print(f"--------------------------------")
print(f"Initial Price: ${initial_price:.2f}")
print(f"Final Price: ${final_price:.2f}")
print(f"Total Return: {buy_hold_return:.2f}%")
print(f"Final Capital with Buy & Hold: ${initial_capital * (1 + buy_hold_return/100):,.2f}")

# Compare with Supertrend Strategy
print(f"\nStrategy Comparison:")
print(f"--------------------------------")
print(f"ATR Trailing Stop Strategy Return: {final_return:.2f}%")
print(f"Buy & Hold Return: {buy_hold_return:.2f}%")
print(f"Outperformance: {final_return - buy_hold_return:.2f}%")


Buy and Hold Strategy:
--------------------------------
Initial Price: $4.08
Final Price: $227.86
Total Return: 5483.36%
Final Capital with Buy & Hold: $5,583,361.93

Strategy Comparison:
--------------------------------
ATR Trailing Stop Strategy Return: 940.69%
Buy & Hold Return: 5483.36%
Outperformance: -4542.67%


# Parameter Optimization
Now let's optimize the Supertrend parameters by testing different combinations of periods and multipliers.

In [11]:
# Function to backtest Supertrend strategy with different parameters
def backtest_supertrend(df, period, multiplier):
    # Create quotes for the indicators
    quotes = [
        Quote(d, o, h, l, c, v)
        for d, o, h, l, c, v in zip(
            df['Date'],
            df['Open'],
            df['High'],
            df['Low'],
            df['Close'],
            df['Volume']
        )
    ]
    
    # Calculate Supertrend
    df_test = df.copy()
    df_test['atr_stop'] = [r.atr_stop for r in indicators.get_atr_stop(quotes)]
    df_test['atr_stop_direction'] = 0.0
    df_test['atr_stop_direction'] = np.where(df_test['atr_stop'] > df_test['Close'], 0.0, 1.0)
    df_test['Signal'] = df_test['atr_stop_direction'].diff()

    # Initialize variables
    capital = 100000
    current_position = 0
    entry_price = 0
    
    # Perform backtesting
    for i in range(1, len(df_test)):
        signal = df_test.iloc[i]['Signal']
        current_price = df_test.iloc[i]['Close']
        
        # Buy Signal
        if signal == 1 and current_position == 0:
            current_position = 1
            entry_price = current_price
        
        # Sell Signal
        elif signal == -1 and current_position == 1:
            exit_price = current_price
            returns = (exit_price - entry_price) / entry_price
            capital *= (1 + returns)
            current_position = 0
    
    final_return = (capital / 100000 - 1) * 100
    return final_return

In [12]:
# # Grid search parameters
# periods = range(5, 31, 2)  # Test periods from 5 to 30 in steps of 2
# multipliers = [x/10 for x in range(10, 51, 5)]  # Test multipliers from 1 to 5 in steps of 0.5

# print("Starting optimization...")
# print(f"Testing {len(periods)} different periods")
# print(f"Testing {len(multipliers)} different multipliers")
# print(f"Total combinations to test: {len(periods) * len(multipliers)}")

# # Store results
# results = []

# # Perform grid search with progress tracking
# total_combinations = len(periods) * len(multipliers)
# current = 0

# for period in periods:
#     for multiplier in multipliers:
#         current += 1
#         if current % 10 == 0:  # Show progress every 10 combinations
#             print(f"Progress: {current}/{total_combinations} combinations tested")
            
#         return_pct = backtest_supertrend(df, period, multiplier)
#         results.append({
#             'period': period,
#             'multiplier': multiplier,
#             'return': return_pct
#         })
        
# # Convert results to DataFrame
# results_df = pd.DataFrame(results)

# # Find best parameters
# best_result = results_df.loc[results_df['return'].idxmax()]
# print("\nOptimization Complete!")
# print("\nBest Parameters Found:")
# print(f"Period: {best_result['period']}")
# print(f"Multiplier: {best_result['multiplier']}")
# print(f"Return: {best_result['return']:.2f}%")

In [13]:
# # Visualize results with heatmap
# # Pivot the results for the heatmap
# heatmap_data = results_df.pivot(index='period', columns='multiplier', values='return')

# # Create heatmap
# plt.figure(figsize=(15, 10))
# sns.heatmap(heatmap_data, 
#             annot=True, 
#             fmt='.1f', 
#             cmap='RdYlGn',
#             center=0,
#             cbar_kws={'label': 'Return %'})
# plt.title('Supertrend Strategy Returns by Parameters')
# plt.xlabel('Multiplier')
# plt.ylabel('Period')
# plt.show()

# # Update the strategy with optimal parameters
# print("\nUpdating strategy with optimal parameters...")
# df['supertrend'] = [r.super_trend for r in indicators.get_super_trend(quotes, int(best_result['period']), best_result['multiplier'])]
# df['supertrend_direction'] = 0.0
# df['supertrend_direction'] = np.where(df['supertrend'] > df['Close'], 0.0, 1.0)
# df['crossover_supertrend'] = df['supertrend_direction'].diff()

In [14]:
df['atr_stop'] = df['atr_stop'].astype(str)

# Now you can serialize your DataFrame
df.to_json() 

'{"Date":{"0":1262563200000,"1":1262649600000,"2":1262736000000,"3":1262822400000,"4":1262908800000,"5":1263168000000,"6":1263254400000,"7":1263340800000,"8":1263427200000,"9":1263513600000,"10":1263859200000,"11":1263945600000,"12":1264032000000,"13":1264118400000,"14":1264377600000,"15":1264464000000,"16":1264550400000,"17":1264636800000,"18":1264723200000,"19":1264982400000,"20":1265068800000,"21":1265155200000,"22":1265241600000,"23":1265328000000,"24":1265587200000,"25":1265673600000,"26":1265760000000,"27":1265846400000,"28":1265932800000,"29":1266278400000,"30":1266364800000,"31":1266451200000,"32":1266537600000,"33":1266796800000,"34":1266883200000,"35":1266969600000,"36":1267056000000,"37":1267142400000,"38":1267401600000,"39":1267488000000,"40":1267574400000,"41":1267660800000,"42":1267747200000,"43":1268006400000,"44":1268092800000,"45":1268179200000,"46":1268265600000,"47":1268352000000,"48":1268611200000,"49":1268697600000,"50":1268784000000,"51":1268870400000,"52":1268956

In [15]:
# ATR Trailing Stop
if __name__ == '__main__':

    chart = Chart(title="ATR Trailing Stop", maximize=True)
    chart.legend(visible=True, color_based_on_candle=True)
    # chart.layout(background_color="white")

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for ATR Trailing Stop

    atr_stop_line = chart.create_line('atr_stop', color='#ffeb3b', width=1, price_line=False, price_label=False)
    atr_stop_line.set(df[['Date', 'atr_stop']])

    # Initialize a list to hold the markers
    markers = []

    # Iterate through the DataFrame to find crossover points
    for i in range(1, len(df)):

        atr_stop_diff = df.iloc[i]['crossover_ATRStop']
        
        current_time = df.iloc[i]['Date']

        # Check for buy signal (ATR Trailing Stop crossover up df['Close'])
        if atr_stop_diff == 1 :
            markers.append({
                'time': current_time,
                'position': 'below',
                'shape': 'arrow_up',
                'color': '#33de3d',
                'text': 'Buy'
            })

        # Check for sell signal (ATR Trailing Stop crossover down df['Close'])
        elif atr_stop_diff == -1 :
            markers.append({
                'time': current_time,
                'position': 'above',
                'shape': 'arrow_down',
                'color': '#f485fb',
                'text': 'Sell'
            })

    # Add all markers at once. It's more efficient than adding them individually in a loop.
    if markers:
        chart.marker_list(markers)

chart.show(block=True)